# CoALA (Cognitive Architectures for Language Agents) | Frameworks & Meta-Approaches

In [1]:
# CoALA Taxonomy: Compare ReAct, Plan-and-Execute, Memory-Augmented via CoALA framework
from dataclasses import dataclass, field
from typing import List

In [2]:
@dataclass
class MemoryConfig:
    working: str = ""          # Short-term context
    episodic: str = "None"     # Past experience recall
    semantic: str = "None"     # Factual knowledge store
    procedural: str = "None"   # Known procedures / skills

@dataclass
class ActionSpaceConfig:
    internal: List[str] = field(default_factory=list)   # Reasoning actions
    external: List[str] = field(default_factory=list)   # Tool/env actions

@dataclass
class DecisionConfig:
    strategy: str = ""
    planning_horizon: str = ""
    feedback_loop: str = ""

@dataclass
class CoALAProfile:
    name: str
    memory: MemoryConfig
    actions: ActionSpaceConfig
    decision: DecisionConfig
    summary: str = ""

In [3]:
# --- Define 3 architectures through the CoALA lens ---
profiles = [
    CoALAProfile(
        name="ReAct", summary="Interleaves reasoning and acting in a single loop",
        memory=MemoryConfig(working="Thought/Action/Obs trace", episodic="None (stateless)",
            semantic="LLM parametric only", procedural="Fixed: Think->Act->Observe"),
        actions=ActionSpaceConfig(internal=["reason"], external=["search", "lookup", "calculate"]),
        decision=DecisionConfig(strategy="Greedy single-step",
            planning_horizon="One step at a time", feedback_loop="Observation -> next thought"),
    ),
    CoALAProfile(
        name="Plan-and-Execute", summary="Creates full plan first, then executes steps",
        memory=MemoryConfig(working="Plan + progress + results", episodic="Step outcomes for replan",
            semantic="None", procedural="Dynamic per-task plan"),
        actions=ActionSpaceConfig(internal=["plan", "replan"], external=["search", "code_exec", "api_call"]),
        decision=DecisionConfig(strategy="Two-phase: plan then execute",
            planning_horizon="Full task (all steps upfront)", feedback_loop="Failure triggers replanning"),
    ),
    CoALAProfile(
        name="Memory-Augmented", summary="Uses vector DB long-term memory to learn across tasks",
        memory=MemoryConfig(working="Query + retrieved memories", episodic="Vector DB of past traces",
            semantic="Knowledge base of domain facts", procedural="Learned from successful runs"),
        actions=ActionSpaceConfig(internal=["reason", "retrieve_memory", "store_memory", "reflect"],
            external=["search", "calculate", "api_call"]),
        decision=DecisionConfig(strategy="Memory-informed retrieval",
            planning_horizon="Adaptive via past experience", feedback_loop="Outcomes stored for future use"),
    ),
]

In [4]:
# --- Print structured CoALA comparison table ---
def print_comparison(agents: List[CoALAProfile]):
    W = 26  # column width
    sep = "-" * (28 + W * len(agents))
    print(f"\n{'COALA ARCHITECTURE COMPARISON':^{28 + W * len(agents)}}\n{sep}")
    hdr = f"{'Dimension':<28}" + "".join(f"{a.name:<{W}}" for a in agents)
    print(f"{hdr}\n{sep}")
    rows = [
        ("MEMORY", None),
        ("  Working Memory",   [a.memory.working[:W-2] for a in agents]),
        ("  Episodic Memory",  [a.memory.episodic[:W-2] for a in agents]),
        ("  Semantic Memory",  [a.memory.semantic[:W-2] for a in agents]),
        ("  Procedural Memory",[a.memory.procedural[:W-2] for a in agents]),
        ("ACTIONS", None),
        ("  Internal",         [", ".join(a.actions.internal)[:W-2] for a in agents]),
        ("  External",         [", ".join(a.actions.external)[:W-2] for a in agents]),
        ("DECISION MAKING", None),
        ("  Strategy",         [a.decision.strategy[:W-2] for a in agents]),
        ("  Planning Horizon", [a.decision.planning_horizon[:W-2] for a in agents]),
        ("  Feedback Loop",    [a.decision.feedback_loop[:W-2] for a in agents]),
    ]
    for label, vals in rows:
        if vals is None:
            print(f"\n{label}")
        else:
            print(f"{label:<28}" + "".join(f"{v:<{W}}" for v in vals))
    print(sep)

print_comparison(profiles)
print("\nKey Insight: CoALA reveals agents differ in MEMORY depth and DECISION strategy,"
      " while sharing similar external action spaces.")


                                      COALA ARCHITECTURE COMPARISON                                       
----------------------------------------------------------------------------------------------------------
Dimension                   ReAct                     Plan-and-Execute          Memory-Augmented          
----------------------------------------------------------------------------------------------------------

MEMORY
  Working Memory            Thought/Action/Obs trace  Plan + progress + result  Query + retrieved memori  
  Episodic Memory           None (stateless)          Step outcomes for replan  Vector DB of past traces  
  Semantic Memory           LLM parametric only       None                      Knowledge base of domain  
  Procedural Memory         Fixed: Think->Act->Obser  Dynamic per-task plan     Learned from successful   

ACTIONS
  Internal                  reason                    plan, replan              reason, retrieve_memory,  
  External         